[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tulin206/erumhub_deep-learning_2026/blob/main/code/07_Monitor_Training_Weather_Regression_exercise.ipynb)

# Predicting the Sunshine Hours Tomorrow

A regression task on the weather prediction dataset

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# TODO: Load the weather prediction dataset from this URL:
# https://zenodo.org/record/5071376/files/weather_prediction_dataset.csv
# YOUR CODE HERE
df = None

# Display summary statistics of the dataframe
df.describe()

In [ ]:
# TODO: Show the shape (rows, columns) of the dataframe
# YOUR CODE HERE

In [ ]:
# TODO: List all column names in the dataframe
# YOUR CODE HERE

In [ ]:
# TODO: Print the set of unique suffixes in column names
# (exclude 'MONTH' and 'DATE', split each column name on '_' and join parts after the first)
# YOUR CODE HERE

## Feature Selection

We use 3 years of daily data (365 × 3 rows). Drop date-related columns and select the target variable.

In [ ]:
# Set the number of rows to use (3 years of daily data)
nr_rows = 365 * 3

# TODO: Select the feature columns (first nr_rows rows), dropping 'DATE' and 'MONTH'
# YOUR CODE HERE
X_data = None

In [ ]:
# Show the shape of the feature matrix
X_data.shape

In [ ]:
# TODO: Print all column names that contain 'BASEL'
# YOUR CODE HERE

In [ ]:
# TODO: Select the target variable: NEXT day's sunshine hours in Basel
# Hint: use df.loc[1:(nr_rows+1)] and select column 'BASEL_sunshine'
# YOUR CODE HERE
y_data = None

In [ ]:
# Show the data type of the target variable
y_data.dtype

## Data Splitting

Split the data into **training** (70%), **validation** (15%), and **test** (15%) sets.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: Split X_data and y_data into training (70%) and not-training (30%) sets
# Use test_size=0.3 and random_state=20211013
# YOUR CODE HERE
X_train, X_not_train, y_train, y_not_train = None, None, None, None

In [ ]:
# TODO: Split the not-training set into validation and test sets (50/50)
# Use random_state=20211014
# YOUR CODE HERE
X_val, X_test, y_val, y_test = None, None, None, None

In [ ]:
# Print shapes to verify the splits
print(X_train.shape)
print(X_test.shape, X_val.shape)

## Model Building

Build a feedforward neural network with two hidden layers using `keras.Input` and `keras.layers.Dense`.

In [ ]:
from tensorflow import keras

def create_nn():
    # TODO: Define the input layer with the correct number of features
    # YOUR CODE HERE
    inputs = None

    # TODO: Add a Dense hidden layer with 100 units and ReLU activation, named 'dense1'
    # YOUR CODE HERE
    dense1 = None

    # TODO: Add a second Dense hidden layer with 50 units and ReLU activation, named 'dense2'
    # YOUR CODE HERE
    dense2 = None

    # TODO: Add the output layer with 1 unit (no activation — this is a regression task)
    # YOUR CODE HERE
    outputs = None

    # Return the model
    return keras.Model(inputs=inputs, outputs=outputs, name="weather_prediction_model")

In [ ]:
# Create the neural network model and display its architecture
model = create_nn()
model.summary()

In [ ]:
# TODO: Compile the model
# Use: optimizer='adam', loss='mse', and add RootMeanSquaredError as a metric
# YOUR CODE HERE

## Training (without validation)

Train the model for 200 epochs and inspect the training RMSE curve.

In [ ]:
# TODO: Train the model on X_train / y_train
# Use batch_size=32, epochs=200, verbose=2
# Store the result in a variable called 'history'
# YOUR CODE HERE
history = None

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# TODO: Convert history.history to a pandas DataFrame called history_df
# YOUR CODE HERE
history_df = None

# Show available columns
history_df.columns

In [ ]:
# TODO: Plot the training RMSE over epochs using seaborn
# Use the column 'root_mean_squared_error' from history_df
# Label axes: x = 'epochs', y = 'RMSE'
# YOUR CODE HERE

## Evaluation

Compare predictions on the training and test sets with scatter plots.

In [ ]:
# TODO: Generate predictions on the training and test sets
# YOUR CODE HERE
y_train_predict = None
y_test_predict = None

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# TODO: Scatter plot for training set (predicted vs true)
# YOUR CODE HERE
axes[0].set_title("training set")
axes[0].set_xlabel("predicted sunshine hours")
axes[0].set_ylabel("true sunshine hours")

# TODO: Scatter plot for test set (predicted vs true)
# YOUR CODE HERE
axes[1].set_title("test set")
axes[1].set_xlabel("predicted sunshine hours")
axes[1].set_ylabel("true sunshine hours")

In [ ]:
# TODO: Evaluate the model on both training and test sets
# Unpack into loss and RMSE for each set
# YOUR CODE HERE
loss_train, rmse_train = None, None
loss_test, rmse_test = None, None

print("training set", rmse_train)
print("    test set", rmse_test)

# Yikes — Overfitting!

The training RMSE is much lower than the test RMSE. Let's investigate and fix this.

## Set Expectations: How Difficult Is the Problem?

Before trying to fix overfitting, let's check how hard the problem is by comparing against a **baseline**: using *yesterday's* sunshine hours to predict *today's*.

In [ ]:
# TODO: Use yesterday's BASEL_sunshine column from X_test as the baseline predictor
# YOUR CODE HERE
y_baseline_prediction = None

plt.figure(figsize=(5, 5), dpi=100)
plt.scatter(y_baseline_prediction, y_test, s=10, alpha=.5)
plt.xlabel("sunshine hours yesterday (baseline)")
plt.ylabel("true sunshine hours")

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# TODO: Calculate RMSE for the neural network predictions on the test set
# YOUR CODE HERE
rmse_nn = None

# TODO: Calculate RMSE for the baseline predictions
# YOUR CODE HERE
rmse_baseline = None

print("neural network RMSE:", rmse_nn)
print("   baseline RMSE:   ", rmse_baseline)

## Counteract Overfitting

### Strategy 1: Reduce model complexity

Redefine `create_nn` to accept `nodes1` and `nodes2` as parameters, then try a smaller network.

In [ ]:
# TODO: Redefine create_nn to accept nodes1 and nodes2 as arguments
# The architecture should be the same as before, but use nodes1 and nodes2 for the hidden layers
# YOUR CODE HERE
def create_nn(nodes1, nodes2):
    pass

In [ ]:
# TODO: Create a smaller model with 10 and 5 nodes, compile it, and train for 200 epochs
# Include validation_data=(X_val, y_val) this time
# Plot both training and validation RMSE after training
# YOUR CODE HERE

### Strategy 2: Early Stopping

Use a larger network (100, 50 nodes) but stop training early when the validation loss stops improving.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# TODO: Create the larger model with create_nn(100, 50) and compile it
# YOUR CODE HERE
model = None

# TODO: Define an EarlyStopping callback
# Monitor 'val_loss', patience=10, verbose=1
# YOUR CODE HERE
earlystop = None

# TODO: Train the model for up to 200 epochs with the EarlyStopping callback
# Include validation_data=(X_val, y_val)
# YOUR CODE HERE
history = None

In [ ]:
# TODO: Convert history.history to a DataFrame and plot
# both 'root_mean_squared_error' and 'val_root_mean_squared_error'
# Label axes: x = 'epochs', y = 'RMSE'
# YOUR CODE HERE

## Further Techniques to Explore

- **Batch Normalisation** — normalise layer inputs to stabilise and speed up training.
- **Dropout layers** — randomly drop units during training to reduce overfitting.

Try adding these to `create_nn` and observe the effect on the validation RMSE!